MD# Lab 9.2: Recourse for Declined Applicants

Chapter 9 data case study. Train a credit model on German Credit, find the cheapest change that would turn each declined applicant into an approved one, first with no constraints and then with only actionable changes, and compare with DiCE. Run the cells in order; checkpoints tell you what you should see.

Data: German Credit (Statlog), 1,000 applicants, from OpenML (`credit-g`); see `data/german-credit.md` for the data card and license. Amounts are in Deutsche Mark (DM).

MD## 1. Model and declined applicants

**Checkpoint 1:** test AUC about 0.85; 87 of 300 test applicants declined at the 0.6 threshold.

In [1]:
import itertools, warnings
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

def load_german():
    """German Credit (Statlog) from OpenML; falls back to data/german_credit.csv if offline."""
    try:
        from sklearn.datasets import fetch_openml
        df = fetch_openml("credit-g", version=1, as_frame=True).frame.copy()
    except Exception:
        df = pd.read_csv("../../../data/german_credit.csv")
    y = (df.pop("class").astype(str) == "good").astype(int)
    for c in df.columns:
        if str(df[c].dtype) in ("category", "object"):
            df[c] = df[c].astype(str)
    return df, y

X, y = load_german()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
CATS = [c for c in X.columns if X[c].dtype == object]

def make_model(seed=0):
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CATS)], remainder="passthrough")
    return make_pipeline(pre, HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                                             max_iter=200, random_state=seed))

THR = 0.6                      # approve when the predicted probability of "good" is at least 0.6
model = make_model().fit(Xtr, ytr)
p = model.predict_proba(Xte)[:, 1]
declined = Xte[p < THR]
print(f"test AUC {roc_auc_score(yte, p):.3f}; declined {len(declined)} of {len(Xte)}")

test AUC 0.847; declined 87 of 300


MD## 2. Cost and action sets

The cost of a change is a range-normalized distance: shortening a loan by 6 months costs 6/68 = 0.088, because loan terms in the training data run from 4 to 72 months. The *free* set lets every feature move; the *actionable* set fixes the features an applicant cannot change today (age, sex and marital status, foreign worker, employment history and others) and allows only sensible directions (savings up, amount down, and so on). Two moves in the actionable set are possible but are not improvements: closing a checking account and registering a telephone. Watch whether they appear.

In [2]:
ORDERED = {"checking_status": ["<0", "0<=X<200", ">=200"],
           "savings_status": ["<100", "100<=X<500", "500<=X<1000", ">=1000"],
           "employment": ["unemployed", "<1", "1<=X<4", "4<=X<7", ">=7"]}
IMMUTABLE = ["age", "personal_status", "foreign_worker", "num_dependents", "credit_history", "purpose",
             "job", "housing", "property_magnitude", "residence_since", "employment"]
RANGE = {c: float(Xtr[c].max() - Xtr[c].min()) for c in X.columns if X[c].dtype != object}

def cost(row, changes):
    """Range-normalized distance: numeric |change| / range; ordered category steps / (levels - 1); other 1."""
    total = 0.0
    for f, v in changes.items():
        if f in RANGE:
            total += abs(v - row[f]) / RANGE[f]
        elif f in ORDERED and row[f] in ORDERED[f] and v in ORDERED[f]:
            lv = ORDERED[f]; total += abs(lv.index(v) - lv.index(row[f])) / (len(lv) - 1)
        else:
            total += 1.0
    return total

def moves(row, free):
    """Candidate one-feature changes. free=True: anything; free=False: the actionable set."""
    out = []
    for f in X.columns:
        old = row[f]
        if free:
            if X[f].dtype == object:
                out += [(f, v) for v in sorted(Xtr[f].unique()) if v != old]
            elif f == "credit_amount":
                out += [(f, round(old * k)) for k in (0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.2, 1.5)]
            else:
                st = {"duration": 6, "age": 5}.get(f, 1)
                out += [(f, old + k * st) for k in range(-4, 5) if k and Xtr[f].min() <= old + k * st <= Xtr[f].max()]
            continue
        if f in IMMUTABLE:
            continue
        if f == "credit_amount":
            out += [(f, round(old * k)) for k in (0.4, 0.5, 0.6, 0.7, 0.8, 0.9)]
        elif f == "duration":
            out += [(f, old + d) for d in (-24, -18, -12, -6, 6, 12) if 4 <= old + d <= 72]
        elif f in ("installment_commitment", "existing_credits"):
            out += [(f, old - d) for d in (1, 2, 3) if old - d >= 1]
        elif f in ("checking_status", "savings_status"):
            lv = ORDERED[f]
            if old in lv:
                out += [(f, v) for v in lv[lv.index(old) + 1:]]
            if f == "checking_status" and old != "no checking":
                out.append((f, "no checking"))        # closing the account: possible, but not an improvement
        elif f == "other_payment_plans" and old != "none":
            out.append((f, "none"))
        elif f == "other_parties" and old == "none":
            out += [(f, "guarantor"), (f, "co applicant")]
        elif f == "own_telephone" and old == "none":
            out.append((f, "yes"))                     # cheap and possible, but not an improvement
    return out

def search(mdl, row, free, thr=THR, n=1):
    """Exhaustive over one and two changes, and three changes among the 12 strongest single moves."""
    c = moves(row, free)
    def batch(combos):
        R = pd.DataFrame([row.to_dict()] * len(combos))
        for i, combo in enumerate(combos):
            for f, v in combo:
                R.at[i, f] = v
        return mdl.predict_proba(R[X.columns])[:, 1]
    one = [(mv,) for mv in c]
    p1 = batch(one)
    top = np.argsort(-p1)[:12]
    two = [(c[i], c[j]) for i, j in itertools.combinations(range(len(c)), 2) if c[i][0] != c[j][0]]
    three = [tuple(c[i] for i in t) for t in itertools.combinations(top, 3) if len({c[i][0] for i in t}) == 3]
    allc = one + two + three
    pp = np.concatenate([p1, batch(two), batch(three)])
    valid = sorted(((cost(row, dict(cb)), cb, q) for cb, q in zip(allc, pp) if q >= thr), key=lambda t: t[0])
    return valid[:n] if n else valid

MD## 3. Unconstrained counterfactuals versus actionable recourse

This cell takes about one minute.

**Checkpoint 2:** 15 of 87 unconstrained counterfactuals change a feature the applicant cannot change now (age or employment history); none of the actionable ones does. Loan duration is the most used feature in both.

In [3]:
free = [search(model, r, True)[0] for _, r in declined.iterrows()]
act = [search(model, r, False)[0] for _, r in declined.iterrows()]
touch = sum(any(f in IMMUTABLE for f, _ in o[1]) for o in free)
print("unconstrained counterfactuals that touch an unchangeable feature:", touch, "of", len(declined))
for name, res in (("free", free), ("actionable", act)):
    used = pd.Series([f for o in res for f, _ in o[1]]).value_counts()
    print(f"\n{name}: median cost {np.median([o[0] for o in res]):.3f}")
    print(used.to_string())

unconstrained counterfactuals that touch an unchangeable feature: 15 of 87

free: median cost 0.178
duration                  56
credit_amount             33
employment                 8
age                        8
checking_status            6
savings_status             6
installment_commitment     1
other_parties              1

actionable: median cost 0.203
duration            64
credit_amount       31
savings_status      15
checking_status      8
existing_credits     1


MD## 4. Four routes for one applicant

One counterfactual hides the choice. Here are four low-cost routes with different feature sets for one declined applicant (the same applicant as the chapter's figure).

**Checkpoint 3:** the applicant asked for 3,345 DM over 24 months and has a score of 0.594; the four routes are a term of 18 months (cost 0.088), savings one band higher (0.333), an installment rate of 3 instead of 4 (0.333) and a checking balance of 200 DM or more (1.0).

In [4]:
def diverse(row, k=4):
    chosen = []
    for cst, cb, q in search(model, row, False, n=None):
        fs = {f for f, _ in cb}
        if all(not fs >= {f for f, _ in c[1]} for c in chosen):
            chosen.append((cst, cb, q))
        if len(chosen) == k:
            break
    return chosen

who = declined.index[declined["duration"].ge(24)][0]
for i, (_, r) in enumerate(declined.iterrows()):
    opts = diverse(r)
    if len(opts) == 4 and len(set().union(*[{f for f, _ in o[1]} for o in opts])) >= 4 and r["duration"] >= 24:
        who = r.name; break
row = declined.loc[who]
print(row[["duration", "credit_amount", "checking_status", "savings_status", "purpose", "age"]].to_dict())
print("score:", round(float(model.predict_proba(declined.loc[[who]])[:, 1][0]), 3))
for cst, cb, q in diverse(row):
    print(f"cost {cst:.3f}  score {q:.2f}  {dict(cb)}")

{'duration': 24, 'credit_amount': 3345, 'checking_status': '<0', 'savings_status': '<100', 'purpose': 'furniture/equipment', 'age': 39}


score: 0.594


cost 0.088  score 0.62  {'duration': np.int64(18)}
cost 0.333  score 0.71  {'savings_status': '100<=X<500'}
cost 0.333  score 0.61  {'installment_commitment': np.int64(3)}
cost 1.000  score 0.74  {'checking_status': '>=200'}


MD## 5. The same applicant with DiCE

DiCE (Mothilal et al., 2020) generates several diverse counterfactuals. Here it may vary only the actionable features, but it does not know which direction of change is sensible for a category.

**Checkpoint 4:** DiCE returns four counterfactuals, all with a score of at least 0.6. Look for any that move a category in a direction the applicant cannot or should not take (for example to `no known savings` or `no checking`). Exact outputs can differ between DiCE versions.

In [5]:
import dice_ml
train = Xtr.copy(); train["good"] = ytr.values
num = [c for c in X.columns if X[c].dtype != object]
d = dice_ml.Data(dataframe=train, continuous_features=num, outcome_name="good")
exp = dice_ml.Dice(d, dice_ml.Model(model=model, backend="sklearn"), method="random")
vary = ["duration", "credit_amount", "savings_status", "checking_status", "installment_commitment",
        "existing_credits", "other_parties", "other_payment_plans"]
q = declined.loc[[who]]
res = exp.generate_counterfactuals(q, total_CFs=4, desired_class=1, features_to_vary=vary,
        permitted_range={"credit_amount": [int(0.4 * q.credit_amount.iloc[0]), int(q.credit_amount.iloc[0])],
                         "duration": [4, int(q.duration.iloc[0])]},
        stopping_threshold=THR, random_seed=0)
cfs = res.cf_examples_list[0].final_cfs_df
print(q[vary].to_string())
print(cfs[vary].to_string())
print("scores:", model.predict_proba(cfs[X.columns])[:, 1].round(2))

     duration  credit_amount savings_status checking_status  installment_commitment  existing_credits other_parties other_payment_plans
919        24           3345           <100              <0                       4                 1          none                none
   duration  credit_amount    savings_status checking_status  installment_commitment  existing_credits other_parties other_payment_plans
0        24           2450  no known savings              <0                       4                 1          none                none
1         6           3345              <100           >=200                       4                 1          none                none
2        24           3345              <100              <0                       2                 1     guarantor                none
3        24           3345              <100     no checking                       4                 2          none                none
scores: [0.68 0.87 0.87 0.78]


MD## 6. Who pays more for recourse?

Compare the cost of the cheapest actionable recourse across groups. The groups are small, so treat differences as a signal to investigate, not a finding.

**Checkpoint 5:** median cost 0.265 for the 32 declined applicants recorded as female, against 0.176 for the 55 others.

In [6]:
ca = np.array([o[0] for o in act])
fem = declined["personal_status"].str.startswith("female").values
young = (declined["age"] < 25).values
print(f"female  : median {np.median(ca[fem]):.3f} (n={fem.sum()})   others: {np.median(ca[~fem]):.3f} (n={(~fem).sum()})")
print(f"under 25: median {np.median(ca[young]):.3f} (n={young.sum()})   others: {np.median(ca[~young]):.3f}")

female  : median 0.265 (n=32)   others: 0.176 (n=55)
under 25: median 0.176 (n=19)   others: 0.203


MD## 7. Does the recourse survive one causal link?

Shortening a loan without changing its amount raises the monthly payment. We assume a simple structural equation: the installment rate (1 to 4, a share of disposable income) scales with amount divided by duration. Apply it to each actionable recourse and score again.

**Checkpoint 6:** 63 of 87 recourses shorten the loan; the installment rate rises for 21; the model still approves 86 of 87. The model hardly reacts to a higher payment burden, which is a question about the model and the world, not about the search.

In [7]:
rows, rises, short = [], 0, 0
for (_, r), o in zip(declined.iterrows(), act):
    ch = dict(o[1]); rr = r.copy()
    for f, v in ch.items():
        rr[f] = v
    short += ch.get("duration", r["duration"]) < r["duration"]
    if "installment_commitment" not in ch:
        ic = r["installment_commitment"] * (ch.get("credit_amount", r["credit_amount"]) / r["credit_amount"]) \
             * (r["duration"] / ch.get("duration", r["duration"]))
        rr["installment_commitment"] = int(np.clip(np.round(ic), 1, 4))
    rises += rr["installment_commitment"] > r["installment_commitment"]
    rows.append(rr)
pc = model.predict_proba(pd.DataFrame(rows)[X.columns])[:, 1]
print("shorten the loan:", short, "| installment rate rises:", rises, "| still approved:", (pc >= THR).sum(), "of", len(act))

shorten the loan: 63 | installment rate rises: 21 | still approved: 86 of 87


MD## 8. Your turn

Remove the two non-improving moves (closing the checking account, registering a telephone) from `moves` and rerun sections 3 and 6. Then make `duration` changeable only upward (a longer loan) and rerun. Write two sentences on how each change to the action set moves the cost of recourse and who bears it.